# 06 – Mô hình cây / boosting và phân tích lỗi

**Người thực hiện:** nhan (Mai Tấn Thiện Nhân)  
**Mục tiêu:** So sánh Decision Tree, Random Forest, XGBoost thật với nhiều tham số; phân tích lỗi mô hình cuối; tạo hình mẫu đầu vào → đầu ra cho slide.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> nhan`.

**Các bước:**
1. `pip install xgboost` rồi chạy XGBoost thật (thay HistGradientBoosting giữa kỳ).
2. Decision Tree: quét max_depth.
3. Random Forest: quét n_estimators, max_features.
4. XGBoost: quét max_depth, learning_rate, n_estimators; thử số chiều SVD.
5. Phân tích lỗi mô hình Danh chọn: đọc 30 mẫu sai, phân loại nguyên nhân.
6. Hình mẫu đầu vào → đầu ra cho slide.

In [ ]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Decision Tree – max_depth

In [ ]:
res, _ = run(member='nhan', model='dt', features='svd', svd=150, params={'max_depth': 20}, exp_name='dt_d20', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 2. Random Forest

In [ ]:
res, _ = run(member='nhan', model='rf', features='svd', params={'n_estimators': 100}, exp_name='rf_100', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. XGBoost thật

In [ ]:
res, _ = run(member='nhan', model='xgb', features='svd', svd=100,
             params={'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}, exp_name='xgb_base', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. Phân tích lỗi
Điền cấu hình cuối cùng mà nhóm chọn (hỏi Danh).

In [ ]:
from sklearn.model_selection import train_test_split
df = load_data(); df['clean'] = df.title.map(clean_title)
X, y = df[['clean','merchant_id']], df.category_label.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
res, pipe = run(member='nhan', model='svc', features='full', params={'C': 1.0}, exp_name='error_analysis', show_report=False)
err = df.loc[Xte.index].assign(pred=pipe.predict(Xte))
err = err[err.category_label != err.pred]
err[['title','category_label','pred']].head(30)

**Tự phân loại 30 lỗi trên thành nhóm nguyên nhân** (chỉ có mã model / sản phẩm lai / tiếng Đức / nhãn sai…) và đếm:

| Nhóm nguyên nhân | Số mẫu | Ví dụ |
|---|---|---|
| … | … | … |

## 5. Hình mẫu đầu vào → đầu ra (cho slide)

In [ ]:
samples = Xte.sample(6, random_state=1)
pred = pipe.predict(samples)
fig, ax = plt.subplots(figsize=(10, 3.4)); ax.axis('off'); ax.set_ylim(0, 1)
for i, ((idx, r), p) in enumerate(zip(samples.iterrows(), pred)):
    true = df.loc[idx, 'category_label']; ok = '✓' if p == true else '✗'
    ax.text(0.0, 0.86 - i * 0.155, df.loc[idx, 'title'][:60], fontsize=10, family='monospace')
    ax.text(0.68, 0.86 - i * 0.155, f'→ {p}  {ok}', fontsize=10, color='green' if p == true else 'red')
ax.set_title('Đầu vào (tiêu đề sản phẩm) → Đầu ra (ngành dự đoán)')
save_fig(fig, 'nhan_sample_io', notebook='06_Nhan_Cay_Boosting_PhanTichLoi.ipynb', member='nhan', cell='Mục 5',
         y_nghia='(tự viết)')

## 6. Kết luận của Nhân (tự viết)